## Libraries and Setup

In [36]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import re
import contractions
from nltk.downloader import download
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from collections import Counter
from numpy.linalg import norm

In [3]:
download('popular')

[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading package inaugural to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package inaugural is already up-to-date!
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /home/jeffmoe/nltk_data...
[nltk_data]    |   Package movie_reviews is already

True

## Text Preprocessing

In [23]:
def preprocess(text:str | list ) -> tuple[list, list]:
    '''
    Function to preprocess the text for vectorization.
    Returns:
        (word_tokens, sentence_tokens)
    '''
    try:
        # check the input for dtype. Raise error if not string or list
        if isinstance(text, list):
            text = ' '.join(text)
        elif not isinstance(text, str):
            raise ValueError("Input must be a list or string")
        text = contractions.fix(text) # Expand contractions first
        stop_words = set(stopwords.words('english')) # init stop words list from NLTK
        lemmatizer = WordNetLemmatizer() # init lemmatizer from NLTK

        tokens_sentences = nltk.sent_tokenize(text)
        normalized_tokens_sentences =[]
        for sentence in tokens_sentences:
            word = nltk.word_tokenize(sentence)
            word = [
                re.sub(r'[^a-zA-Z0-9]', '', w.lower())
                for w in word
            ] # normalize
            word = [w for w in word if w and w not in stop_words] # remove stop words
            word = [lemmatizer.lemmatize(w) for w in word] # reduce words to base values
            normalized_tokens_sentences.append(' '.join(word)) # combine back into sentence structure

        

        normalized_tokens_words = nltk.word_tokenize(text)
        normalized_tokens_words = [
            re.sub(r'[^a-zA-Z0-9]', '', w.lower()) 
            for w in normalized_tokens_words
            ] # normalize
        normalized_tokens_words = [w for w in normalized_tokens_words if w and w not in stop_words] # remove stop words
        normalized_tokens_words = [lemmatizer.lemmatize(w) for w in normalized_tokens_words] # reduce words to base values
        

        
        return normalized_tokens_words, normalized_tokens_sentences
    except Exception as e:
        print(f"Error in preprocesssing: {e}")
        return [], []

In [10]:
sample_text_1 = """
Steam is a premier digital distribution platform for PC games, developed by Valve. 
It functions as a storefront, community hub, and library, allowing users to buy, play, create, and discuss video games. 
It also offers robust cloud saves, achievements, and multiplayer matchmaking."""
sample_text_2 = """
Steam is water in its gaseous state (water vapor). 
While pure steam is an invisible gas, the term is commonly used to describe the visible 
white mist of condensed water droplets that forms when hot water vapor meets cooler air."""
sample_text_3 = """
Xbox Game Pass is a rotating video game subscription service by Microsoft that allows 
members to download or stream hundreds of games across 
Xbox consoles, PC, and cloud-compatible devices for a flat monthly fee."""
sample_text_4 = """
A car (or automobile) is a wheeled, self-propelled motor vehicle designed primarily for personal transport on roads. 
It typically seats one to eight people, relies on an internal combustion engine, 
electric motor, or hybrid system, and uses four wheels to move."""

In [35]:
print(f"Sample 1: {preprocess(sample_text_1)}")
print(f"Sample 2: {preprocess(sample_text_2)}")
print(f"Sample 3: {preprocess(sample_text_3)}")
print(f"Sample 4: {preprocess(sample_text_4)}")

processed_1 = preprocess(sample_text_1)
processed_2 = preprocess(sample_text_2)
processed_3 = preprocess(sample_text_3)
processed_4 = preprocess(sample_text_4)

combined_docs = []
combined_docs = [processed_1[0],processed_2[0],processed_3[0],processed_4[0]]
combined_docs

Sample 1: (['steam', 'premier', 'digital', 'distribution', 'platform', 'pc', 'game', 'developed', 'valve', 'function', 'storefront', 'community', 'hub', 'library', 'allowing', 'user', 'buy', 'play', 'create', 'discus', 'video', 'game', 'also', 'offer', 'robust', 'cloud', 'save', 'achievement', 'multiplayer', 'matchmaking'], ['steam premier digital distribution platform pc game developed valve', 'function storefront community hub library allowing user buy play create discus video game', 'also offer robust cloud save achievement multiplayer matchmaking'])
Sample 2: (['steam', 'water', 'gaseous', 'state', 'water', 'vapor', 'pure', 'steam', 'invisible', 'gas', 'term', 'commonly', 'used', 'describe', 'visible', 'white', 'mist', 'condensed', 'water', 'droplet', 'form', 'hot', 'water', 'vapor', 'meet', 'cooler', 'air'], ['steam water gaseous state water vapor', 'pure steam invisible gas term commonly used describe visible white mist condensed water droplet form hot water vapor meet cooler air

[['steam',
  'premier',
  'digital',
  'distribution',
  'platform',
  'pc',
  'game',
  'developed',
  'valve',
  'function',
  'storefront',
  'community',
  'hub',
  'library',
  'allowing',
  'user',
  'buy',
  'play',
  'create',
  'discus',
  'video',
  'game',
  'also',
  'offer',
  'robust',
  'cloud',
  'save',
  'achievement',
  'multiplayer',
  'matchmaking'],
 ['steam',
  'water',
  'gaseous',
  'state',
  'water',
  'vapor',
  'pure',
  'steam',
  'invisible',
  'gas',
  'term',
  'commonly',
  'used',
  'describe',
  'visible',
  'white',
  'mist',
  'condensed',
  'water',
  'droplet',
  'form',
  'hot',
  'water',
  'vapor',
  'meet',
  'cooler',
  'air'],
 ['xbox',
  'game',
  'pas',
  'rotating',
  'video',
  'game',
  'subscription',
  'service',
  'microsoft',
  'allows',
  'member',
  'download',
  'stream',
  'hundred',
  'game',
  'across',
  'xbox',
  'console',
  'pc',
  'cloudcompatible',
  'device',
  'flat',
  'monthly',
  'fee'],
 ['car',
  'automobile',
  

## Vectorization and Cosine Similarity

In [46]:
def cosine_sim(token: list | tuple) -> list:
    '''
    Function to use TF-IDF to vectorize tokens + apply cosine similarity
    Returns:
        (cos_sim_matrix)
    '''
    token = [' '.join(t) for t in token] # join tokens back into strings for vectorization
    vectorizer = TfidfVectorizer()
    X_sklearn = vectorizer.fit_transform(token)

    cos_sim_sklearn = cosine_similarity(X_sklearn)
    return cos_sim_sklearn


In [38]:
def manual_cosine_sim(token: list | tuple) -> list:
    '''
    Function to use bag of words to manually vectorize tokens + apply cosine similarity
    Returns:
        (cos_sim_matrix)
    '''
    text = [' '.join(t) for t in text]
    vocab = sorted(set(word for text in text for word in doc))
    vocab_index = {word: i for i, word in enumerate(vocab)}

    # Create document-term matrix
    X_manual = np.zeros((len(text), len(vocab)))

    for i, doc in enumerate(text):
        counts = Counter(doc)
        for word, count in counts.items():
            X_manual[i, vocab_index[word]] = count
    
    sim_matrix = np.zeros((X_manual.shape[0], X_manual.shape[0]))
    for i in range(X_manual.shape[0]):
        for j in range(X_manual.shape[0]):
            dot = np.dot(X_manual[i], X_manual[j])
            sim_matrix[i, j] = dot / (norm(X_manual[i]) * norm(X_manual[j])) if norm(X_manual[i]) != 0 and norm(X_manual[j]) != 0 else 0
    return sim_matrix


In [47]:
cosine_sim(combined_docs)

array([[1.        , 0.03562629, 0.17396747, 0.        ],
       [0.03562629, 1.        , 0.        , 0.        ],
       [0.17396747, 0.        , 1.        , 0.        ],
       [0.        , 0.        , 0.        , 1.        ]])

## Visualizing Vectors and Matrices

In [ ]:
def graph_vectors(A: list,B: list):

    plt.figure()
    sns.heatmap(A, annot=True, cmap="coolwarm")
    plt.title("Sklearn Cosine Similarity")
    plt.show()

    plt.figure()
    sns.heatmap(B, annot=True, cmap="coolwarm")
    plt.title("Manual Cosine Similarity")
    plt.show()

    X_dense = A.toarray()
    pca = PCA(n_components=2)
    X_2D = pca.fit_transform(X_dense)

    # Plot
    plt.figure()
    for i, point in enumerate(X_2D):
        plt.scatter(point[0], point[1])
        plt.text(point[0], point[1], f"Doc {i}", fontsize=10)

    plt.title("Document Vector Visualization (PCA)")
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.show()

